# 03. Buffer Feature Engineering
**목적**: 전력설비 중심 multi-scale buffer를 생성하고, 각 buffer 안에서 공간 특성 feature를 산출한다.

## 산출 Feature 목록

| Feature | 데이터 소스 | 등급 |
|---------|------------|------|
| `facility_density_{r}m` | 날씨마루 전력설비 | L1 |
| `nearest_facility_dist` | 날씨마루 전력설비 | L1 |
| `forest_ratio_{r}m` | 산림청 임상도 | L1 |
| `conifer_ratio_{r}m` | 산림청 임상도 | L1 |
| `distance_to_forest` | 산림청 임상도 | L1 |
| `slope_mean_{r}m` | 국가공간정보포털 DEM | L1 ★ |
| `elevation_mean_{r}m` | 국가공간정보포털 DEM | L1 |
| `aspect_risk_{r}m` | 국가공간정보포털 DEM | L1 |
| `fire_count_{r}m` | 산림청 산불통계 | L1 |
| `elec_fire_count_{r}m` | 소방청 화재통계 | L1 |

> **slope** 는 국제 SHAP 연구에서 산불위험 예측 1위 변수

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')

from config import (
    DATA_PROCESSED, CRS_PROJ, BUFFER_RADII, EXTERNAL_FILES
)

with open(DATA_PROCESSED / 'column_map.json', encoding='utf-8') as f:
    cmap = json.load(f)
FLAGS = cmap['flags']
FC = cmap['facility']
FID_COL = FC['facility_id']

gdf_fac = gpd.read_file(DATA_PROCESSED / 'facility_proj.gpkg')
print(f'설비 수: {len(gdf_fac):,}  |  CRS: {gdf_fac.crs}')

## 1. Multi-scale Buffer 생성

In [ ]:
buffers = {}
for r in BUFFER_RADII:
    buffers[r] = gdf_fac[[FID_COL, 'geometry']].copy()
    buffers[r]['geometry'] = gdf_fac.geometry.buffer(r)
    print(f'  buffer {r:>5}m 생성 완료')

## 2. 설비 밀도 + 최근접 거리 (날씨마루 전력설비)

In [ ]:
def compute_facility_density(gdf_fac, buf_gdf, fid_col, radius):
    joined = gpd.sjoin(
        gdf_fac[[fid_col, 'geometry']],
        buf_gdf[[fid_col, 'geometry']].rename(columns={fid_col: f'{fid_col}_buf'}),
        how='right', predicate='within'
    )
    joined = joined[joined[fid_col] != joined[f'{fid_col}_buf']]
    cnt = joined.groupby(f'{fid_col}_buf').size().reset_index()
    cnt.columns = [fid_col, f'facility_density_{radius}m']
    return cnt

df_density = gdf_fac[[FID_COL]].copy()
for r in [500, 1000]:
    d = compute_facility_density(gdf_fac, buffers[r], FID_COL, r)
    df_density = df_density.merge(d, on=FID_COL, how='left')
    df_density[f'facility_density_{r}m'] = df_density[f'facility_density_{r}m'].fillna(0).astype(int)

# 최근접 거리
coords = np.column_stack([gdf_fac.geometry.x, gdf_fac.geometry.y])
tree = cKDTree(coords)
dists, _ = tree.query(coords, k=2)
df_density['nearest_facility_dist_m'] = dists[:, 1]

print('설비 밀도 feature 완료')
print(df_density.describe())

## 3. 산림 Feature — 임상도 (산림청)

임상도 컬럼:
- `FRTP_CD`: 1=침엽수, 2=활엽수, 3=혼효림, 4=죽림, 5=무립목지
- `DENS_CD`: 밀도 (1=소, 2=중, 3=밀)

In [ ]:
FOREST_FILE = EXTERNAL_FILES['forest_gpkg']
df_forest = gdf_fac[[FID_COL]].copy()

if FOREST_FILE.exists():
    gdf_forest = gpd.read_file(FOREST_FILE).to_crs(CRS_PROJ)
    print(f'임상도 로드: {len(gdf_forest):,}개 폴리곤  |  컬럼: {gdf_forest.columns.tolist()}')

    # 침엽수 / 전체 산림 분리
    gdf_conifer = gdf_forest[gdf_forest['FRTP_CD'] == 1] if 'FRTP_CD' in gdf_forest.columns else gdf_forest
    gdf_all_forest = gdf_forest[gdf_forest['FRTP_CD'].isin([1,2,3,4])] if 'FRTP_CD' in gdf_forest.columns else gdf_forest

    for r in [250, 500, 1000]:
        buf = buffers[r]
        buf_area = np.pi * r ** 2

        # 전체 산림 비율
        inter = gpd.overlay(buf[[FID_COL, 'geometry']], gdf_all_forest[['geometry']], how='intersection')
        inter['inter_area'] = inter.geometry.area
        s = inter.groupby(FID_COL)['inter_area'].sum().reset_index()
        s[f'forest_ratio_{r}m'] = (s['inter_area'] / buf_area).clip(0, 1)
        df_forest = df_forest.merge(s[[FID_COL, f'forest_ratio_{r}m']], on=FID_COL, how='left')
        df_forest[f'forest_ratio_{r}m'] = df_forest[f'forest_ratio_{r}m'].fillna(0)

        # 침엽수 비율
        inter_c = gpd.overlay(buf[[FID_COL, 'geometry']], gdf_conifer[['geometry']], how='intersection')
        inter_c['inter_area'] = inter_c.geometry.area
        sc = inter_c.groupby(FID_COL)['inter_area'].sum().reset_index()
        sc[f'conifer_ratio_{r}m'] = (sc['inter_area'] / buf_area).clip(0, 1)
        df_forest = df_forest.merge(sc[[FID_COL, f'conifer_ratio_{r}m']], on=FID_COL, how='left')
        df_forest[f'conifer_ratio_{r}m'] = df_forest[f'conifer_ratio_{r}m'].fillna(0)

    # 최근접 산림까지 거리
    df_forest['distance_to_forest_m'] = gdf_fac.geometry.apply(
        lambda g: gdf_all_forest.distance(g).min()
    ).values

    print('\n산림 feature 생성 완료')
    print(df_forest.describe())
else:
    print(f'[미수집] {FOREST_FILE}')
    print('→ map.forest.go.kr 에서 임상도 신청 후 data/external/forest.gpkg 로 저장')
    for r in [250, 500, 1000]:
        df_forest[f'forest_ratio_{r}m'] = np.nan
        df_forest[f'conifer_ratio_{r}m'] = np.nan
    df_forest['distance_to_forest_m'] = np.nan

## 4. 지형 Feature — DEM (국가공간정보포털)

Slope는 국제 SHAP 연구에서 산불위험 예측 **1위 변수** (SHAP=1.62)

- `slope_mean_{r}m` : 평균 경사도
- `elevation_mean_{r}m` : 평균 고도
- `aspect_risk_{r}m` : 사면 방향 위험도 (남향=고위험)

In [ ]:
import rasterio
from rasterio.features import geometry_window
from rasterio.transform import rowcol
from rasterio.warp import reproject, Resampling

DEM_FILE = EXTERNAL_FILES['dem_tif']
df_terrain = gdf_fac[[FID_COL]].copy()

def compute_slope_aspect(dem_array, transform, nodata=-9999):
    """DEM 배열에서 경사도(slope)와 사면방향(aspect) 계산."""
    from numpy import gradient, arctan2, sqrt, degrees, pi
    dem = dem_array.astype(float)
    dem[dem == nodata] = np.nan
    cell_size = abs(transform.a)  # 픽셀 크기(미터)
    dy, dx = gradient(dem, cell_size)
    slope = degrees(arctan2(sqrt(dx**2 + dy**2), 1))
    aspect = degrees(arctan2(dx, -dy)) % 360
    # 사면 방향 위험도: 남향(180°) 기준, 건조하고 일사량 많은 방향
    aspect_risk = 1 - abs(aspect - 180) / 180  # 0~1, 남향=1
    return slope, aspect_risk

if DEM_FILE.exists():
    with rasterio.open(DEM_FILE) as src:
        dem_data = src.read(1)
        dem_transform = src.transform
        dem_crs = src.crs
        nodata_val = src.nodata or -9999

    slope_arr, aspect_arr = compute_slope_aspect(dem_data, dem_transform, nodata_val)

    # 설비 위치에서 buffer 내 DEM 통계 추출
    from rasterio.mask import mask as rio_mask
    from shapely.geometry import mapping

    # CRS 통일
    gdf_fac_dem = gdf_fac.to_crs(dem_crs) if gdf_fac.crs != dem_crs else gdf_fac

    slope_results = {}
    elevation_results = {}
    aspect_results = {}

    for r in [500, 1000]:
        buf = gdf_fac_dem.copy()
        buf['geometry'] = buf.geometry.buffer(r)

        slopes, elevs, aspects = [], [], []
        for _, row in buf.iterrows():
            try:
                dem_crop, _ = rio_mask(rasterio.open(DEM_FILE), [mapping(row.geometry)],
                                       crop=True, filled=True, fill_value=np.nan)
                arr = dem_crop[0].astype(float)
                arr[arr == nodata_val] = np.nan
                sl, asp = compute_slope_aspect(arr, dem_transform, nodata_val)
                slopes.append(np.nanmean(sl))
                elevs.append(np.nanmean(arr))
                aspects.append(np.nanmean(asp))
            except Exception:
                slopes.append(np.nan)
                elevs.append(np.nan)
                aspects.append(np.nan)

        df_terrain[f'slope_mean_{r}m']     = slopes
        df_terrain[f'elevation_mean_{r}m'] = elevs
        df_terrain[f'aspect_risk_{r}m']    = aspects

    print('지형 feature 생성 완료')
    print(df_terrain.describe())
else:
    print(f'[미수집] {DEM_FILE}')
    print('→ 국가공간정보포털 또는 SRTM 30m 다운로드 후 data/external/dem.tif 로 저장')
    for r in [500, 1000]:
        df_terrain[f'slope_mean_{r}m']     = np.nan
        df_terrain[f'elevation_mean_{r}m'] = np.nan
        df_terrain[f'aspect_risk_{r}m']    = np.nan

## 5. 과거 화재 Prior — 산불통계 (산림청) + 소방청 전기화재

In [ ]:
df_prior = gdf_fac[[FID_COL]].copy()

# ── 5-A. 산림청 산불발생통계 ──────────────────────────────────────────────
FIRE_FOREST = EXTERNAL_FILES['fire_history_csv']
if FIRE_FOREST.exists():
    df_fire_f = pd.read_csv(FIRE_FOREST, encoding='utf-8-sig')
    print(f'산불통계 로드: {len(df_fire_f):,}건  컬럼: {df_fire_f.columns.tolist()}')

    # 위도/경도 컬럼 자동 탐색
    lat_c = next((c for c in df_fire_f.columns if '위도' in c or 'lat' in c.lower()), None)
    lon_c = next((c for c in df_fire_f.columns if '경도' in c or 'lon' in c.lower()), None)

    if lat_c and lon_c:
        gdf_fire_f = gpd.GeoDataFrame(
            df_fire_f,
            geometry=gpd.points_from_xy(df_fire_f[lon_c], df_fire_f[lat_c]),
            crs='EPSG:4326'
        ).to_crs(CRS_PROJ)

        for r in [1000, 3000, 5000]:
            buf_r = buffers[min(r, max(BUFFER_RADII))]
            joined = gpd.sjoin(gdf_fire_f[['geometry']], buf_r[[FID_COL, 'geometry']],
                               how='right', predicate='within')
            cnt = joined.groupby(FID_COL).size().reset_index(name=f'fire_count_{r}m')
            df_prior = df_prior.merge(cnt, on=FID_COL, how='left')
            df_prior[f'fire_count_{r}m'] = df_prior[f'fire_count_{r}m'].fillna(0).astype(int)
        print('  산불 prior feature 생성 완료')
    else:
        print('  위도/경도 컬럼 없음 — 주소 기반 지오코딩 필요')
        for r in [1000, 3000, 5000]:
            df_prior[f'fire_count_{r}m'] = np.nan
else:
    print(f'[미수집] {FIRE_FOREST} → data.go.kr 에서 산림청_산불통계데이터 다운로드')
    for r in [1000, 3000, 5000]:
        df_prior[f'fire_count_{r}m'] = np.nan

# ── 5-B. 소방청 화재통계 (전기적 원인) ────────────────────────────────────
FIRE_SOS = EXTERNAL_FILES['fire_sos_csv']
if FIRE_SOS.exists():
    df_fire_s = pd.read_csv(FIRE_SOS, encoding='utf-8-sig')
    print(f'\n소방청 화재통계 로드: {len(df_fire_s):,}건  컬럼: {df_fire_s.columns.tolist()}')

    # 전기적 원인 필터링
    elec_keywords = ['전기', '전선', '누전', '합선', '과부하', '배선']
    cause_col = next((c for c in df_fire_s.columns if '원인' in c or 'cause' in c.lower()), None)
    if cause_col:
        elec_mask = df_fire_s[cause_col].astype(str).str.contains('|'.join(elec_keywords), na=False)
        df_elec = df_fire_s[elec_mask].copy()
        print(f'  전기적 원인 화재: {len(df_elec):,}건 ({len(df_elec)/len(df_fire_s)*100:.1f}%)')
    else:
        df_elec = df_fire_s.copy()

    lat_c = next((c for c in df_elec.columns if '위도' in c or 'lat' in c.lower()), None)
    lon_c = next((c for c in df_elec.columns if '경도' in c or 'lon' in c.lower()), None)

    if lat_c and lon_c:
        gdf_elec = gpd.GeoDataFrame(
            df_elec,
            geometry=gpd.points_from_xy(df_elec[lon_c], df_elec[lat_c]),
            crs='EPSG:4326'
        ).to_crs(CRS_PROJ)

        for r in [500, 1000]:
            buf_r = buffers[min(r, max(BUFFER_RADII))]
            joined = gpd.sjoin(gdf_elec[['geometry']], buf_r[[FID_COL, 'geometry']],
                               how='right', predicate='within')
            cnt = joined.groupby(FID_COL).size().reset_index(name=f'elec_fire_count_{r}m')
            df_prior = df_prior.merge(cnt, on=FID_COL, how='left')
            df_prior[f'elec_fire_count_{r}m'] = df_prior[f'elec_fire_count_{r}m'].fillna(0).astype(int)
        print('  소방청 전기화재 prior feature 생성 완료')
    else:
        print('  위도/경도 컬럼 없음')
        for r in [500, 1000]:
            df_prior[f'elec_fire_count_{r}m'] = np.nan
else:
    print(f'[미수집] {FIRE_SOS} → data.go.kr 에서 소방청 화재발생현황 다운로드')
    for r in [500, 1000]:
        df_prior[f'elec_fire_count_{r}m'] = np.nan

print('\n=== Prior feature 현황 ===')
print(df_prior.isnull().sum())

## 6. 전체 Buffer Feature 통합 저장

In [ ]:
df_buf = df_density.copy()
df_buf = df_buf.merge(df_forest,  on=FID_COL, how='left')
df_buf = df_buf.merge(df_terrain, on=FID_COL, how='left')
df_buf = df_buf.merge(df_prior,   on=FID_COL, how='left')

# 수집된 feature vs NaN 현황 요약
n_features = df_buf.shape[1] - 1
nan_cols = df_buf.isnull().any()
collected = (~nan_cols).sum() - 1  # FID 제외
pending   = nan_cols.sum()

print(f'전체 feature 수 : {n_features}')
print(f'수집 완료       : {collected}개')
print(f'데이터 대기 중  : {pending}개')
print()
if pending > 0:
    print('대기 중 feature (날씨마루 승인 또는 외부 데이터 다운로드 후 채워짐):')
    print(df_buf.columns[nan_cols].tolist())

df_buf.to_parquet(DATA_PROCESSED / 'buffer_features.parquet', index=False)
print('\n저장 완료: buffer_features.parquet')
print('다음 단계: 04_weather_feature_engineering.ipynb')